In [6]:
from __future__ import annotations
import os
from io import StringIO
from pathlib import Path
from typing import Dict, Any, List, Optional
from src.config import get_secret
import numpy as np
import pandas as pd
import requests


# ----------------------------
# Global Config
# ----------------------------
YEAR: int = 2026
TOUR_KEYS: List[str] = ["PGA", "EURO", "LIV"]

# DataGolf endpoints
DG_ROUNDS_URL: str = "https://feeds.datagolf.com/historical-raw-data/rounds"
DG_FIELD_URL: str  = "https://feeds.datagolf.com/field-updates"
DG_ODDS_URL: str   = "https://feeds.datagolf.com/betting-tools/outrights"

# API key (DO NOT hardcode in notebook)
DG_API_KEY = get_secret("DATAGOLF_API_KEY")
if not DG_API_KEY:
    raise RuntimeError("Missing DATAGOLF_API_KEY env var. Set it and rerun.")

# Project paths (absolute to your repo)
PROJECT_ROOT: Path = Path("/Users/joshmacbook/python_projects/OAD")
DATA_ROOT: Path    = PROJECT_ROOT / "Data"
RAW_DIR: Path      = DATA_ROOT / "Raw"
CLEAN_DIR: Path    = DATA_ROOT / "Clean" / "Combined"
INUSE_DIR: Path    = DATA_ROOT / "in Use"

# Create dirs
for _p in [RAW_DIR, CLEAN_DIR, INUSE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# Canonical output paths (things we DO update)
CLEAN_YEAR_PATH: Path = CLEAN_DIR / f"combined_rounds_{YEAR}.csv"
INUSE_ALL_PATH: Path  = INUSE_DIR / "combined_rounds_all_2017_2026.csv"
MST_24P_PATH: Path    = INUSE_DIR / "combined_roundlevel_2024_present.csv"
FINISHES_PATH: Path   = INUSE_DIR / "Finishes.csv"

# Weekly field/odds artifacts (PGA only for field/owgr + odds)
THIS_WEEK_FIELD_CSV: Path = INUSE_DIR / "this_week_field.csv"
THIS_WEEK_ODDS_CSV: Path  = INUSE_DIR / "this_week_odds.csv"

FIELDS_XLSX: Path       = INUSE_DIR / "Fields.xlsx"
ALL_PLAYERS_XLSX: Path  = INUSE_DIR / "All_players.xlsx"
SCHED_XLSX: Path        = INUSE_DIR / "OAD_2026_Schedule.xlsx"

# Things we explicitly DO NOT touch in the weekly refresh notebook
MST_17_23_EVENTMEAN_PATH: Path = INUSE_DIR / "combined_eventlevel_pga_2017_2023_mean.csv"

# ----------------------------
# Tour Mapping
# ----------------------------
# Keys are your internal tour keys; api_tour is what DataGolf expects.
TOURS: Dict[str, Dict[str, str]] = {
    "PGA":  {"folder": "PGA",  "prefix": "PGA",  "api_tour": "pga"},
    "EURO": {"folder": "EURO", "prefix": "EURO", "api_tour": "euro"},
    "LIV":  {"folder": "LIV",  "prefix": "liv",  "api_tour": "liv"},
}

# ----------------------------
# Requests session
# ----------------------------
TIMEOUT: int = 60
SESSION = requests.Session()

# ----------------------------
# Small Utilities
# ----------------------------
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def coerce_int(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df

def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def pull_csv_to_df(url: str, params: Dict[str, Any], *, save_path: Optional[Path] = None) -> pd.DataFrame:
    resp = SESSION.get(url, params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    if save_path is not None:
        ensure_dir(save_path.parent)
        df.to_csv(save_path, index=False)
        print(f"[PULL] Saved → {save_path} | rows: {len(df):,}")
    return df

def read_excel(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_excel(path)

def write_excel(path: Path, df: pd.DataFrame, *, sheet_name: str = "Sheet1") -> None:
    ensure_dir(path.parent)
    with pd.ExcelWriter(path, engine="openpyxl", mode="w") as w:
        df.to_excel(w, index=False, sheet_name=sheet_name)

def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, **kwargs)

def write_csv(path: Path, df: pd.DataFrame) -> None:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)

# ----------------------------
# DataGolf Pullers
# ----------------------------
def pull_rounds_year(*, tour_key: str, year: int = YEAR) -> Path:
    if tour_key not in TOURS:
        raise ValueError(f"Unknown tour_key: {tour_key}. Expected one of {list(TOURS)}")

    cfg = TOURS[tour_key]
    out_dir = ensure_dir(RAW_DIR / cfg["folder"])
    out_path = out_dir / f"{cfg['prefix']}_rounds_{year}.csv"

    params = {
        "tour": cfg["api_tour"],
        "event_id": "all",
        "year": year,
        "file_format": "csv",
        "key": DG_API_KEY,
    }

    df = pull_csv_to_df(DG_ROUNDS_URL, params, save_path=out_path)
    df = coerce_int(df, ["event_id", "dg_id", "year", "round_num"])
    df.to_csv(out_path, index=False)

    print(f"[ROUNDS] {tour_key} {year} → {out_path} | rows: {len(df):,}")
    return out_path

def pull_this_week_field_pga(*, save_path: Path = THIS_WEEK_FIELD_CSV) -> pd.DataFrame:
    params = {"tour": "pga", "file_format": "csv", "key": DG_API_KEY}
    df = pull_csv_to_df(DG_FIELD_URL, params, save_path=save_path)
    df = coerce_int(df, ["event_id", "dg_id", "year", "round_num"])
    return df

def pull_this_week_odds_pga(*, save_path: Path = THIS_WEEK_ODDS_CSV) -> pd.DataFrame:
    params = {"tour": "pga", "file_format": "csv", "key": DG_API_KEY}
    df = pull_csv_to_df(DG_ODDS_URL, params, save_path=save_path)
    df = coerce_int(df, ["event_id", "dg_id", "year"])
    return df

# ----------------------------
# Cleaning helpers (finish_num)
# ----------------------------
NON_NUMERIC_FINISH = {"CUT", "WD", "DQ", "MDF", "NAN"}

def clean_finish(val):
    v = str(val).upper().strip()
    if v in NON_NUMERIC_FINISH:
        return v
    if v.startswith("T") and v[1:].isdigit():
        return int(v[1:])
    if v.isdigit():
        return int(v)
    return v

# ----------------------------
# LIV patch map (event_id + course_num)
# ----------------------------
LIV_EVENT_ID_MAP_RAW = {
    "adelaide":                         1012,
    "andalucia":                        1024,
    "bangkok":                          1006,
    "bedminster":                       1003,
    "boston":                           1004,
    "chicago":                          1005,
    "dallas":                           1031,
    "dallas (team finalstroke play)":   1026,
    "dc":                               1015,
    "greenbrier":                       1017,
    "hong kong":                        1020,
    "houston":                          1022,
    "indianapolis":                     1032,
    "jeddah":                           1007,
    "korea":                            1029,
    "las vegas":                        1019,
    "london":                           1001,
    "mayakoba":                         1009,
    "mexico city":                      1028,
    "miami":                            1021,
    "miami (team finalstroke play)":    1008,
    "michigan (team finalstroke play)": 1033,
    "nashville":                        1023,
    "orlando":                          1011,
    "portland":                         1002,
    "promotions event":                 1018,
    "riyadh":                           1027,
    "singapore":                        1013,
    "tucson":                           1010,
    "tulsa":                            1014,
    "united kingdom":                   1025,
    "valderrama":                       1016,
    "virginia":                         1030,
}
LIV_EVENT_ID_MAP = {k.strip().lower(): v for k, v in LIV_EVENT_ID_MAP_RAW.items()}

def patch_liv_ids(df: pd.DataFrame) -> pd.DataFrame:
    if not {"tour", "event_name"}.issubset(df.columns):
        return df

    out = df.copy()
    out["tour_clean"] = out["tour"].astype(str).str.strip().str.upper()
    out["event_name_clean"] = out["event_name"].astype(str).str.strip().str.lower()

    liv_mask = out["tour_clean"].eq("LIV")
    in_map = out["event_name_clean"].isin(LIV_EVENT_ID_MAP.keys())
    target = liv_mask & in_map

    if target.any():
        new_ids = out.loc[target, "event_name_clean"].map(LIV_EVENT_ID_MAP).astype("Int64")
        if "event_id" in out.columns:
            out.loc[target, "event_id"] = new_ids
        if "course_num" in out.columns:
            out.loc[target, "course_num"] = new_ids
        print(f"[PATCH] LIV id/course_num updates: {int(target.sum()):,}")

    return out.drop(columns=["tour_clean", "event_name_clean"], errors="ignore")

# ----------------------------
# round_date derivation from event_completed + round_num
# ----------------------------
def add_round_date(df: pd.DataFrame) -> pd.DataFrame:
    needed = {"event_completed", "round_num", "tour", "event_name"}
    if not needed.issubset(df.columns):
        return df

    out = df.copy()
    out["event_completed_dt"] = safe_to_datetime(out["event_completed"])
    if out["event_completed_dt"].isna().all():
        return df

    out["round_num"] = pd.to_numeric(out["round_num"], errors="coerce")
    group_cols = ["tour", "event_name", "event_completed_dt"]
    max_round = out.groupby(group_cols)["round_num"].transform("max")
    offset_days = max_round - out["round_num"]

    out["round_date_dt"] = out["event_completed_dt"] - pd.to_timedelta(offset_days, unit="D")
    out["round_date"] = out["round_date_dt"].dt.date.astype(str)

    return out.drop(columns=["event_completed_dt", "round_date_dt"], errors="ignore")

# ----------------------------
# Safe replace helpers (surgical 2026-only updates)
# ----------------------------
def replace_year_in_csv(existing_path: Path, new_year_df: pd.DataFrame, *, year_col: str = "year", year_val: int = YEAR) -> None:
    if existing_path.exists():
        old = pd.read_csv(existing_path)
        old[year_col] = pd.to_numeric(old.get(year_col), errors="coerce")
        kept = old[old[year_col] != year_val].copy()
        out = pd.concat([kept, new_year_df], ignore_index=True)
        out.to_csv(existing_path, index=False)
        print(f"[REPLACE] {existing_path} | dropped {year_val}, appended {year_val} | rows: {len(out):,}")
    else:
        new_year_df.to_csv(existing_path, index=False)
        print(f"[WRITE] {existing_path} | rows: {len(new_year_df):,}")

# ----------------------------
# Finishes builder (2026-only)
# ----------------------------
def build_finishes_2026_from_roundlevel(roundlevel_24p_path: Path = MST_24P_PATH, out_path: Path = FINISHES_PATH) -> None:
    df = pd.read_csv(roundlevel_24p_path)

    # derive year
    if "season" in df.columns:
        df["year"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    else:
        if "round_date" not in df.columns:
            raise RuntimeError("Cannot derive year: no 'season' and no 'round_date' in roundlevel file.")
        df["round_date"] = pd.to_datetime(df["round_date"], errors="coerce")
        df["year"] = df["round_date"].dt.year.astype("Int64")

    df = df[df["year"] == YEAR].copy()

    df["event_id"] = pd.to_numeric(df.get("event_id"), errors="coerce").astype("Int64")
    df["dg_id"]    = pd.to_numeric(df.get("dg_id"), errors="coerce").astype("Int64")
    df["finish_num"] = pd.to_numeric(df.get("finish_num"), errors="coerce")

    sort_cols = [c for c in ["year", "event_id", "dg_id", "round_num", "round_date"] if c in df.columns]
    df_sorted = df.sort_values(sort_cols)

    final = (
        df_sorted
        .groupby(["year", "event_id", "dg_id"], as_index=False)
        .tail(1)
        .copy()
    )

    final["made_cut"] = final["finish_num"].notna().astype(int)
    final["CUT"] = (1 - final["made_cut"]).astype(int)

    fn = final["finish_num"]
    final["win"]    = ((fn == 1)  & fn.notna()).astype(int)
    final["top_5"]  = ((fn <= 5)  & fn.notna()).astype(int)
    final["top_10"] = ((fn <= 10) & fn.notna()).astype(int)
    final["top_25"] = ((fn <= 25) & fn.notna()).astype(int)

    if "finish_text" not in final.columns and "fin_text" in final.columns:
        final["finish_text"] = final["fin_text"]

    cols = [
        "year", "event_name", "event_id", "event_completed",
        "player_name", "dg_id", "finish_text", "finish_num",
        "win", "top_5", "top_10", "top_25", "made_cut", "CUT"
    ]
    for c in cols:
        if c not in final.columns:
            final[c] = pd.NA

    out = final[cols].sort_values(["event_id", "finish_num", "player_name"], na_position="last")
    out.to_csv(out_path, index=False)
    print(f"[FINISHES] Wrote {len(out):,} rows for {YEAR} → {out_path}")


In [7]:
# ============================================================
# BLOCK 1: Weekly PGA Field + Odds → update Fields.xlsx + All_players.xlsx
# ============================================================

# 1) Pull this week field (PGA) and keep only needed columns
field_raw = pull_this_week_field_pga(save_path=THIS_WEEK_FIELD_CSV)

need_field = ["event_name", "event_id", "dg_id", "player_name", "owgr_rank"]
missing = [c for c in need_field if c not in field_raw.columns]
if missing:
    raise RuntimeError(f"Field feed missing columns: {missing}. Got: {list(field_raw.columns)}")

field = field_raw[need_field].copy()
field = coerce_int(field, ["event_id", "dg_id", "year", "round_num"])
field["owgr_rank"] = pd.to_numeric(field["owgr_rank"], errors="coerce")
field = field.dropna(subset=["event_id", "dg_id", "player_name"]).copy()
field["player_name"] = field["player_name"].astype(str)
field["event_name"]  = field["event_name"].astype(str)

event_ids = sorted(field["event_id"].dropna().unique().tolist())
if len(event_ids) != 1:
    raise RuntimeError(f"Expected exactly 1 event_id in field feed; got {event_ids}")
THIS_EVENT_ID = int(event_ids[0])
THIS_EVENT_NAME = field["event_name"].iloc[0]

print(f"[FIELD] event_id={THIS_EVENT_ID} | event_name={THIS_EVENT_NAME} | players={len(field):,}")

# 2) Pull this week odds (PGA) and build dg_id -> close_odds map
try:
    odds_raw = pull_this_week_odds_pga(save_path=THIS_WEEK_ODDS_CSV)
except Exception as e:
    # fallback to existing odds file if the endpoint params ever change
    if THIS_WEEK_ODDS_CSV.exists():
        print("[ODDS] pull failed; falling back to existing file:", THIS_WEEK_ODDS_CSV)
        odds_raw = pd.read_csv(THIS_WEEK_ODDS_CSV)
    else:
        raise RuntimeError(f"[ODDS] pull failed and no existing file found. Error: {e}") from e

odds_raw = coerce_int(odds_raw, ["event_id", "dg_id", "year"])

if "dg_id" not in odds_raw.columns:
    raise RuntimeError(f"Odds missing dg_id. Got columns: {list(odds_raw.columns)}")
if "datagolf_base_history_fit" not in odds_raw.columns:
    raise RuntimeError(
        "Odds missing datagolf_base_history_fit (needed for close_odds). "
        f"Got columns: {list(odds_raw.columns)}"
    )

odds = odds_raw[["dg_id", "datagolf_base_history_fit"]].copy()
odds["dg_id"] = pd.to_numeric(odds["dg_id"], errors="coerce").astype("Int64")
odds["datagolf_base_history_fit"] = pd.to_numeric(odds["datagolf_base_history_fit"], errors="coerce")
odds = odds.dropna(subset=["dg_id"]).copy()

close_odds_by_id = dict(zip(odds["dg_id"].astype(int), odds["datagolf_base_history_fit"]))
print(f"[ODDS] rows={len(odds):,} | mapped_ids={len(close_odds_by_id):,}")

# 3) Get schedule start_date for this event_id and store it into Fields.xlsx "event_completed"
sched = read_excel(SCHED_XLSX)
for c in ["event_id", "start_date"]:
    if c not in sched.columns:
        raise RuntimeError(f"Schedule missing '{c}'. Got columns: {list(sched.columns)}")

sched["event_id"] = pd.to_numeric(sched["event_id"], errors="coerce").astype("Int64")
sched["start_date"] = pd.to_datetime(sched["start_date"], errors="coerce")

row = sched.loc[sched["event_id"] == THIS_EVENT_ID]
if row.empty:
    raise RuntimeError(f"Schedule has no row for event_id={THIS_EVENT_ID}")

start_date_value = pd.to_datetime(row["start_date"].iloc[0])
if pd.isna(start_date_value):
    raise RuntimeError(f"Schedule start_date is NaT for event_id={THIS_EVENT_ID}")

print(f"[SCHED] start_date={start_date_value} (stored into Fields.xlsx.event_completed)")

# 4) Update Fields.xlsx idempotently on (year, event_id, dg_id)
fields_existing = read_excel(FIELDS_XLSX)

required_cols = ["year", "event_name", "event_id", "event_completed", "player_name", "dg_id", "close_odds"]
for c in required_cols:
    if c not in fields_existing.columns:
        raise RuntimeError(f"Fields.xlsx missing '{c}'. Got columns: {list(fields_existing.columns)}")

fields_existing["year"] = pd.to_numeric(fields_existing["year"], errors="coerce").astype("Int64")
fields_existing["event_id"] = pd.to_numeric(fields_existing["event_id"], errors="coerce").astype("Int64")
fields_existing["dg_id"] = pd.to_numeric(fields_existing["dg_id"], errors="coerce").astype("Int64")

wk = field[["event_name", "event_id", "dg_id", "player_name"]].copy()
wk["year"] = YEAR
wk["event_completed"] = start_date_value
wk["close_odds"] = wk["dg_id"].astype(int).map(close_odds_by_id)
wk = wk[required_cols].copy()

key_cols = ["year", "event_id", "dg_id"]
existing_keys = set(
    map(tuple,
        fields_existing[key_cols]
        .dropna()
        .astype({"year": "int64", "event_id": "int64", "dg_id": "int64"})
        .to_numpy()
    )
)
wk_keys = set(
    map(tuple,
        wk[key_cols]
        .dropna()
        .astype({"year": "int64", "event_id": "int64", "dg_id": "int64"})
        .to_numpy()
    )
)

overlap = existing_keys.intersection(wk_keys)

fields_keep = fields_existing
if overlap:
    mask_overlap = fields_keep.apply(
        lambda r: (
            (int(r["year"]), int(r["event_id"]), int(r["dg_id"])) in overlap
            if pd.notna(r["year"]) and pd.notna(r["event_id"]) and pd.notna(r["dg_id"])
            else False
        ),
        axis=1,
    )
    fields_keep = fields_keep.loc[~mask_overlap].copy()

fields_new = pd.concat([fields_keep, wk], ignore_index=True)
fields_new["player_name"] = fields_new["player_name"].astype(str)
fields_new["event_name"]  = fields_new["event_name"].astype(str)

write_excel(FIELDS_XLSX, fields_new)
print(f"[FIELDS] updated. existing={len(fields_existing):,} | week={len(wk):,} | final={len(fields_new):,} | replaced={len(overlap):,}")

# 5) Update All_players.xlsx owgr for dg_ids in this week field
ap = read_excel(ALL_PLAYERS_XLSX)
if "dg_id" not in ap.columns:
    raise RuntimeError(f"All_players.xlsx missing dg_id. Got columns: {list(ap.columns)}")
if "owgr" not in ap.columns:
    ap["owgr"] = pd.NA

ap["dg_id"] = pd.to_numeric(ap["dg_id"], errors="coerce").astype("Int64")

owgr_map = dict(
    zip(
        field["dg_id"].dropna().astype(int),
        field["owgr_rank"],
    )
)

before_nonnull = ap["owgr"].notna().sum()
ap["owgr"] = ap.apply(
    lambda r: owgr_map.get(int(r["dg_id"]), r["owgr"]) if pd.notna(r["dg_id"]) else r["owgr"],
    axis=1,
)
after_nonnull = ap["owgr"].notna().sum()

write_excel(ALL_PLAYERS_XLSX, ap)
print(f"[ALL_PLAYERS] owgr updated. nonnull_before={before_nonnull:,} | nonnull_after={after_nonnull:,} | updated_ids={len(owgr_map):,}")


[PULL] Saved → /Users/joshmacbook/python_projects/OAD/Data/MST/this_week_field.csv | rows: 72
[FIELD] event_id=7 | event_name=The Genesis Invitational | players=72
[ODDS] pull failed; falling back to existing file: /Users/joshmacbook/python_projects/OAD/Data/MST/this_week_odds.csv
[ODDS] rows=72 | mapped_ids=72
[SCHED] start_date=2026-02-19 00:00:00 (stored into Fields.xlsx.event_completed)
[FIELDS] updated. existing=1,210 | week=72 | final=1,210 | replaced=72
[ALL_PLAYERS] owgr updated. nonnull_before=196 | nonnull_after=196 | updated_ids=72


In [8]:
# ============================================================
# BLOCK 2: Weekly rounds refresh (2026 only) for PGA/EURO/LIV
# ============================================================

print("\n[GUARD] NOT touching:", MST_17_23_EVENTMEAN_PATH)

# 1) Pull 2026 rounds for each tour into Raw
for tour_key in TOUR_KEYS:
    pull_rounds_year(tour_key=tour_key, year=YEAR)

# 2) Load only the freshly pulled 2026 files and stack
dfs = []
for tour_key in TOUR_KEYS:
    cfg = TOURS[tour_key]
    fpath = RAW_DIR / cfg["folder"] / f"{cfg['prefix']}_rounds_{YEAR}.csv"
    df = read_csv(fpath)
    df["tour"] = tour_key
    df["year"] = YEAR
    dfs.append(df)

combined_2026 = pd.concat(dfs, ignore_index=True)
combined_2026 = coerce_int(combined_2026, ["event_id", "dg_id", "year", "round_num"])

# 3) finish_num from fin_text (if present)
if "fin_text" in combined_2026.columns:
    combined_2026["fin_text"] = combined_2026["fin_text"].astype(str).str.upper().str.strip()
    combined_2026["finish_num"] = combined_2026["fin_text"].apply(clean_finish)
    print("[CLEAN] finish_num created from fin_text")
else:
    print("[CLEAN] fin_text not present; skipping finish_num creation")

# 4) Patch LIV event_id/course_num (if applicable)
combined_2026 = patch_liv_ids(combined_2026)

# 5) round_date derivation (if applicable)
combined_2026 = add_round_date(combined_2026)

# 6) Write Clean/Combined/combined_rounds_2026.csv
write_csv(CLEAN_YEAR_PATH, combined_2026)
print(f"[CLEAN] wrote {CLEAN_YEAR_PATH} | rows={len(combined_2026):,}")

# 7) Replace 2026 only in the big "in use" combined file
replace_year_in_csv(INUSE_ALL_PATH, combined_2026, year_col="year", year_val=YEAR)

# 8) Replace 2026 only in MST combined_roundlevel_2024_present.csv
#    (keep other >=2024 rows as-is)
if MST_24P_PATH.exists():
    old_24p = read_csv(MST_24P_PATH)
    old_24p["year"] = pd.to_numeric(old_24p.get("year"), errors="coerce")
    keep = old_24p[(old_24p["year"] >= 2024) & (old_24p["year"] != YEAR)].copy()
    new_24p = pd.concat([keep, combined_2026], ignore_index=True)
    write_csv(MST_24P_PATH, new_24p)
    print(f"[MST] updated {MST_24P_PATH} | kept 2024-2025, replaced {YEAR} | rows={len(new_24p):,}")
else:
    # if file doesn't exist, write 2026 only (deterministic)
    write_csv(MST_24P_PATH, combined_2026)
    print(f"[MST] wrote new {MST_24P_PATH} (only {YEAR}) | rows={len(combined_2026):,}")

# 9) Build Finishes.csv (2026 only)
build_finishes_2026_from_roundlevel(MST_24P_PATH, FINISHES_PATH)

print("[DONE] weekly rounds refresh complete.")



[GUARD] NOT touching: /Users/joshmacbook/python_projects/OAD/Data/MST/combined_eventlevel_pga_2017_2023_mean.csv
[PULL] Saved → /Users/joshmacbook/python_projects/OAD/Data/Raw/PGA/PGA_rounds_2026.csv | rows: 2,074
[ROUNDS] PGA 2026 → /Users/joshmacbook/python_projects/OAD/Data/Raw/PGA/PGA_rounds_2026.csv | rows: 2,074
[PULL] Saved → /Users/joshmacbook/python_projects/OAD/Data/Raw/EURO/EURO_rounds_2026.csv | rows: 1,454
[ROUNDS] EURO 2026 → /Users/joshmacbook/python_projects/OAD/Data/Raw/EURO/EURO_rounds_2026.csv | rows: 1,454
[PULL] Saved → /Users/joshmacbook/python_projects/OAD/Data/Raw/LIV/liv_rounds_2026.csv | rows: 607
[ROUNDS] LIV 2026 → /Users/joshmacbook/python_projects/OAD/Data/Raw/LIV/liv_rounds_2026.csv | rows: 607
[CLEAN] finish_num created from fin_text
[PATCH] LIV id/course_num updates: 607
[CLEAN] wrote /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2026.csv | rows=4,135


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_78292/2440789679.py:261: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  old = pd.read_csv(existing_path)


[REPLACE] /Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv | dropped 2026, appended 2026 | rows: 315,175
[MST] updated /Users/joshmacbook/python_projects/OAD/Data/MST/combined_roundlevel_2024_present.csv | kept 2024-2025, replaced 2026 | rows=79,471
[FINISHES] Wrote 1,441 rows for 2026 → /Users/joshmacbook/python_projects/OAD/Data/MST/Finishes.csv
[DONE] weekly rounds refresh complete.
